# Lab 4: Orchestration Patterns

## NovaPay Fraud Detection & Payment Support

**Objective:** Compare and implement three orchestration patterns for multi-agent systems. Learn when to use each pattern for real-world fintech workflows.

**Duration:** ~55 minutes  
**Prerequisites:** Labs 1-3 completed  
**Model:** Claude Haiku via Amazon Bedrock (us-east-1)

### Patterns Covered

1. **Single Agent** — One agent with all tools (baseline)
2. **Sequential Workflow** — Fixed pipeline: check → analyze → act
3. **Orchestrator/Specialist** — One agent delegates to domain experts
4. **Agent-as-a-Tool** — Strands pattern: wrap `Agent()` in a `@tool`


In [ ]:
# ── Install dependencies ────────────────────────────────────────
import subprocess, sys, importlib

print('Installing dependencies...')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install',
     'strands-agents==1.2.0',
     'strands-agents-tools==0.2.3',
     'boto3>=1.34.0',
     '--quiet']
)
importlib.invalidate_caches()
print('✅ Packages installed')

# ── Imports ─────────────────────────────────────────────────────
import json
from datetime import datetime
from IPython.display import HTML, display

from strands import Agent, tool
from strands.models import BedrockModel
from botocore.config import Config as BotocoreConfig

# ── Configure the model ─────────────────────────────────────────
bedrock_model = BedrockModel(
    model_id='us.anthropic.claude-haiku-4-5-20251001-v1:0',
    boto_client_config=BotocoreConfig(
        retries={'max_attempts': 3},
        connect_timeout=5,
        read_timeout=60
    )
)

display(HTML('<h3 style="color: #2ecc71;">✅ Setup complete</h3>'))


In [ ]:
# NovaPay mock data
NOVAPAY_CUSTOMERS = {
    "CUST-1001": {
        "name": "Amara Okafor",
        "balance": 15420.50,
        "currency": "NGN",
        "account_type": "premium",
        "risk_score": 12
    },
    "CUST-1002": {
        "name": "Kwame Mensah",
        "balance": 3200.00,
        "currency": "GHS",
        "account_type": "standard",
        "risk_score": 45
    }
}

NOVAPAY_TRANSACTIONS = {
    "TXN-101": {
        "customer_id": "CUST-1001",
        "amount": 2500.00,
        "currency": "NGN",
        "type": "transfer",
        "status": "completed",
        "timestamp": "2024-01-15T10:30:00Z",
        "recipient": "Chidi Nwankwo",
        "fraud_flags": []
    },
    "TXN-102": {
        "customer_id": "CUST-1001",
        "amount": 89000.00,
        "currency": "NGN",
        "type": "transfer",
        "status": "flagged",
        "timestamp": "2024-01-15T23:45:00Z",
        "recipient": "Unknown Merchant XYZ",
        "fraud_flags": ["unusual_amount", "odd_hour", "new_recipient"]
    },
    "TXN-103": {
        "customer_id": "CUST-1002",
        "amount": 150.00,
        "currency": "GHS",
        "type": "payment",
        "status": "completed",
        "timestamp": "2024-01-16T08:00:00Z",
        "recipient": "Accra Utilities",
        "fraud_flags": []
    }
}

NOVAPAY_POLICIES = {
    "max_single_transfer": 100000.00,
    "daily_limit": 500000.00,
    "high_risk_threshold": 70,
    "fraud_flag_threshold": 2,
    "auto_block_flags": ["unusual_amount", "odd_hour", "new_recipient"]
}

print(f"NovaPay data loaded: {len(NOVAPAY_CUSTOMERS)} customers, {len(NOVAPAY_TRANSACTIONS)} transactions")


## 2. Pattern 0: Single Agent Baseline

The simplest approach: one agent with **ALL** tools.

**Pros:** Simple, no coordination overhead  
**Cons:** Gets confused with too many tools, can't parallelize


In [ ]:
# Define all tools
@tool
def check_balance(customer_id: str) -> str:
    """Check account balance.
    Args:
        customer_id: Customer ID (e.g., CUST-1001)
    """
    customer = NOVAPAY_CUSTOMERS.get(customer_id)
    if not customer:
        return json.dumps({"error": "Customer not found"})
    return json.dumps({"name": customer["name"], "balance": customer["balance"], "currency": customer["currency"]})


@tool
def get_transactions(transaction_id: str) -> str:
    """Get transaction details.
    Args:
        transaction_id: Transaction ID (e.g., TXN-101)
    """
    txn = NOVAPAY_TRANSACTIONS.get(transaction_id)
    if not txn:
        return json.dumps({"error": "Transaction not found"})
    return json.dumps(txn)


@tool
def fraud_check(transaction_id: str) -> str:
    """Run fraud risk assessment.
    Args:
        transaction_id: Transaction to analyze
    """
    txn = NOVAPAY_TRANSACTIONS.get(transaction_id)
    if not txn:
        return json.dumps({"error": "Transaction not found"})
    flags = txn.get("fraud_flags", [])
    risk_score = len(flags) * 30
    return json.dumps({
        "risk_score": risk_score,
        "risk_level": "high" if risk_score >= 60 else "medium" if risk_score >= 30 else "low",
        "flags": flags
    })


@tool
def check_policy(action_type: str, amount: float) -> str:
    """Check if an action complies with NovaPay policies.
    Args:
        action_type: Type of action (transfer, payment, withdrawal)
        amount: Amount to check against policy limits
    """
    if amount > NOVAPAY_POLICIES["max_single_transfer"]:
        return json.dumps({
            "compliant": False,
            "reason": f"Exceeds max single transfer of {NOVAPAY_POLICIES['max_single_transfer']}"
        })
    return json.dumps({"compliant": True, "action": action_type, "amount": amount})


print("✅ 4 tools defined: check_balance, get_transactions, fraud_check, check_policy")


In [ ]:
# Single agent with ALL tools
single_agent = Agent(
    model=bedrock_model,
    tools=[check_balance, get_transactions, fraud_check, check_policy],
    system_prompt="You are NovaPay's universal support agent. Handle all requests."
)

print("✅ Single agent created with 4 tools")
print("\nTesting...")
response = single_agent("Check if TXN-102 is fraudulent and whether its amount complies with our policies.")
print(f"\n🤖 {response}")


## 3. Pattern 1: Sequential Workflow

A **fixed pipeline** where each step feeds into the next:

**Step 1: CHECK** → **Step 2: ANALYZE** → **Step 3: ACT**  
(get data)  →  (assess risk)  →  (decide action)

**Use when:** The workflow is predictable and steps don't need to branch.  
**Example:** KYC verification, loan application processing.


In [ ]:
# Sequential workflow: each step is a separate agent with focused tools

# Step 1: Data Collection Agent
data_collector = Agent(
    model=bedrock_model,
    tools=[check_balance, get_transactions],
    system_prompt="""You are a data collection agent. Your ONLY job is to gather transaction
and customer data. Return raw facts only - no analysis or recommendations.
Format your output as a structured summary."""
)

# Step 2: Risk Analyzer Agent
risk_analyzer = Agent(
    model=bedrock_model,
    tools=[fraud_check, check_policy],
    system_prompt="""You are a risk analysis agent. Given transaction data, assess:
1. Fraud risk level and specific flags
2. Policy compliance status
Output a risk report with scores and flags. Do NOT recommend actions."""
)

# Step 3: Decision Agent (no tools - pure reasoning)
decision_agent = Agent(
    model=bedrock_model,
    tools=[],
    system_prompt="""You are a decision agent. Given a risk report, decide:
- APPROVE: risk is low, proceed normally
- REVIEW: medium risk, flag for human review
- BLOCK: high risk, block immediately

Provide a clear decision with justification."""
)

print("✅ Sequential pipeline created: data_collector → risk_analyzer → decision_agent")


In [ ]:
# Run the sequential pipeline
print("=" * 60)
print("SEQUENTIAL PIPELINE: Investigating TXN-102")
print("=" * 60)

# Step 1: Collect data
print("\n📋 STEP 1: Data Collection")
print("-" * 40)
step1_result = data_collector("Get all details about transaction TXN-102 and the associated customer.")
print(f"{step1_result}")

# Step 2: Analyze risk (pass step 1 output as context)
print("\n🔍 STEP 2: Risk Analysis")
print("-" * 40)
step2_result = risk_analyzer(f"Analyze this transaction for fraud and policy compliance. Transaction ID: TXN-102. Context from data collection: {step1_result}")
print(f"{step2_result}")

# Step 3: Make decision (pass step 2 output)
print("\n⚖️ STEP 3: Decision")
print("-" * 40)
step3_result = decision_agent(f"Based on this risk report, what action should we take? Risk report: {step2_result}")
print(f"{step3_result}")


## 4. Pattern 2: Orchestrator/Specialist

One **orchestrator agent** decides which **specialist agent** to delegate to:

```
ORCHESTRATOR (decides)
    ├── FRAUD SPECIALIST
    ├── ACCOUNT SPECIALIST
    └── PAYMENT SPECIALIST
```

**Use when:** Requests are diverse and need different expertise.  
**Example:** Customer support hub, incident response.


In [ ]:
# Specialist agents (each focused on one domain)
fraud_specialist = Agent(
    model=bedrock_model,
    tools=[fraud_check, get_transactions],
    system_prompt="""You are NovaPay's Fraud Investigation Specialist.
You ONLY handle fraud-related queries. Investigate suspicious transactions,
assess risk levels, and recommend actions (block/monitor/allow)."""
)

account_specialist = Agent(
    model=bedrock_model,
    tools=[check_balance],
    system_prompt="""You are NovaPay's Account Specialist.
You ONLY handle account-related queries: balances, account types, limits.
Be helpful and precise with financial information."""
)

policy_specialist = Agent(
    model=bedrock_model,
    tools=[check_policy],
    system_prompt="""You are NovaPay's Policy Compliance Specialist.
You ONLY handle policy questions: transfer limits, compliance checks.
Cite specific policy rules in your answers."""
)


# Wrap specialists as tools for the orchestrator
@tool
def ask_fraud_specialist(query: str) -> str:
    """Delegate to the Fraud Investigation Specialist for fraud-related questions.
    Use when the user asks about suspicious transactions, fraud checks, or risk assessment.
    Args:
        query: The fraud-related question to investigate
    """
    result = fraud_specialist(query)
    return str(result)


@tool
def ask_account_specialist(query: str) -> str:
    """Delegate to the Account Specialist for account-related questions.
    Use when the user asks about balances, account status, or account types.
    Args:
        query: The account-related question
    """
    result = account_specialist(query)
    return str(result)


@tool
def ask_policy_specialist(query: str) -> str:
    """Delegate to the Policy Compliance Specialist for policy questions.
    Use when the user asks about transfer limits, compliance rules, or policy checks.
    Args:
        query: The policy-related question
    """
    result = policy_specialist(query)
    return str(result)


# Create the orchestrator
orchestrator = Agent(
    model=bedrock_model,
    tools=[ask_fraud_specialist, ask_account_specialist, ask_policy_specialist],
    system_prompt="""You are NovaPay's Support Orchestrator. You do NOT answer questions directly.
Instead, delegate to the appropriate specialist:
- Fraud questions → ask_fraud_specialist
- Account questions → ask_account_specialist
- Policy questions → ask_policy_specialist

For complex queries, you may delegate to multiple specialists and combine their answers."""
)

print("✅ Orchestrator + 3 specialists created")


In [ ]:
# Test the orchestrator pattern
print("=" * 60)
print("ORCHESTRATOR PATTERN: Complex multi-domain query")
print("=" * 60)

response = orchestrator("I'm customer CUST-1001. Check my balance, and also investigate TXN-102 - it looks suspicious.")
print(f"\n🤖 Orchestrator Response:\n{response}")


## 5. Pattern 3: Agent-as-a-Tool (Strands Native Pattern)

The most powerful Strands pattern: wrap an entire `Agent()` inside a `@tool`.  
This gives you **composable agents** — agents that use other agents as tools.

**Key Difference from Orchestrator:**
- Orchestrator: routes to specialists based on intent
- Agent-as-a-Tool: the outer agent can call inner agents as part of multi-step reasoning


In [ ]:
# Agent-as-a-tool pattern: inner agents wrapped in @tool

# Inner agent: fraud investigator
inner_fraud_agent = Agent(
    model=bedrock_model,
    tools=[fraud_check, get_transactions],
    system_prompt="You are a fraud investigator. Analyze transactions and report risk levels."
)

# Inner agent: compliance checker
inner_compliance_agent = Agent(
    model=bedrock_model,
    tools=[check_policy],
    system_prompt="You are a compliance officer. Check transactions against NovaPay policies."
)


# Wrap agents as tools - THIS IS THE KEY PATTERN
@tool
def investigate_fraud(transaction_id: str) -> str:
    """Use the fraud investigation agent to deeply analyze a transaction.
    This agent has access to transaction data and fraud scoring.
    Args:
        transaction_id: The transaction to investigate
    """
    result = inner_fraud_agent(
        f"Investigate transaction {transaction_id} for fraud. Provide risk score and detailed analysis."
    )
    return str(result)


@tool
def verify_compliance(action_type: str, amount: float) -> str:
    """Use the compliance agent to verify a transaction meets all policies.
    Args:
        action_type: Type of financial action (transfer, payment)
        amount: The transaction amount
    """
    result = inner_compliance_agent(
        f"Check if a {action_type} of {amount} complies with NovaPay policies."
    )
    return str(result)


# Outer agent uses inner agents as tools
supervisor_agent = Agent(
    model=bedrock_model,
    tools=[check_balance, investigate_fraud, verify_compliance],
    system_prompt="""You are NovaPay's Senior Support Supervisor.
For simple queries (balance checks), handle directly.
For complex queries, delegate to specialist agents:
- investigate_fraud: deep fraud analysis
- verify_compliance: policy compliance check

Synthesize results from multiple agents into a clear recommendation."""
)

print("✅ Agent-as-a-Tool pattern configured")
print("  Outer agent: supervisor_agent")
print("  Inner agents: fraud investigator, compliance checker")


In [ ]:
# Test agent-as-a-tool
print("=" * 60)
print("AGENT-AS-A-TOOL: Deep investigation with agent delegation")
print("=" * 60)

response = supervisor_agent(
    "Transaction TXN-102 for 89000 NGN was flagged. "
    "Investigate it for fraud AND verify if the amount complies with our policies. "
    "Give me a final recommendation."
)
print(f"\n🤖 Supervisor: {response}")


## 6. Comparison: When to Use Each Pattern

| Pattern | Complexity | Latency | Best For | NovaPay Example |
|---------|-----------|---------|----------|------------------|
| Single Agent | Low | Low | Simple Q&A, ≤5 tools | Customer support chat |
| Sequential | Medium | High (serial) | Fixed pipelines, ETL | KYC verification flow |
| Orchestrator/Specialist | Medium | Medium | Routing diverse requests | Support hub |
| Agent-as-a-Tool | High | High (nested) | Deep multi-step reasoning | Fraud investigation |

### Trade-offs

**Simplicity ←————————————————→ Capability**  
Single Agent → Sequential Pipeline → Orchestrator/Specialist → Agent-as-Tool (Nested)


## 7. Decision Framework for FDEs

Use this flowchart to choose the right pattern:

```
START
├── Q: How many tools? ≤ 5 and single domain?
│   └── YES → Single Agent ✅
├── Q: Is the workflow fixed/predictable?
│   └── YES → Sequential Pipeline ✅
├── Q: Are there distinct domains that rarely overlap?
│   └── YES → Orchestrator/Specialist ✅
└── Q: Do you need agents to reason about other agents' outputs?
    └── YES → Agent-as-a-Tool ✅
```

### Additional Considerations

- **Cost:** More agents = more LLM calls = higher cost
- **Latency:** Nested agents add latency (each is a full LLM round-trip)
- **Debuggability:** Simpler patterns are easier to trace and debug
- **Scalability:** Specialist patterns scale better as features grow


## 8. Practical Exercise: Choose Your Pattern

**Scenario:** NovaPay wants to build an automated system that:

1. Monitors real-time transactions
2. Flags suspicious activity
3. Notifies the fraud team
4. Auto-blocks transactions above risk threshold

Think about which pattern fits best before looking at the solution.


In [ ]:
# Solution: This is a Sequential Pipeline (fixed steps, predictable flow)
# Because the steps are always: Monitor -> Flag -> Notify -> Block

display(HTML('''
<div style="background: #1a1a2e; padding: 20px; border-radius: 10px; color: white;">
<h3>✅ Recommended: Sequential Pipeline</h3>
<p><b>Why?</b></p>
<ul>
<li>Steps are fixed and predictable (always the same order)</li>
<li>Each step's output feeds directly into the next</li>
<li>No branching or dynamic routing needed</li>
<li>Easier to audit for compliance (linear trace)</li>
</ul>
<p><b>If requirements change</b> (e.g., sometimes skip notification),
then Orchestrator pattern becomes better.</p>
</div>
'''))


## 🧠 Knowledge Check

1. What's the key difference between Orchestrator/Specialist and Agent-as-a-Tool?
2. Why would you choose Sequential over a single agent with all tools?
3. In the Agent-as-a-Tool pattern, how does the outer agent "know" about inner agents?
4. What are the cost implications of nested agent patterns?
5. For a real-time payment gateway, which pattern minimizes latency?

### Answers

<details>
<summary>▶ Click to reveal</summary>

1. Orchestrator routes based on intent; Agent-as-a-Tool lets the outer agent reason about inner agents' outputs mid-task.
2. Sequential isolates each step's responsibility, making errors easier to debug and each agent more focused.
3. The outer agent knows about inner agents through the `@tool` docstrings — the description tells it when and how to use each inner agent.
4. Each nested agent = additional LLM API call = higher cost and latency.
5. Single Agent (lowest latency, no coordination overhead).

</details>

---

## ➡️ Next Steps

In Lab 5, you'll learn:
- Amazon Bedrock Knowledge Bases
- RAG (Retrieve-Augmented Generation) patterns
- Building agents with KB-backed retrieval tools

---

*NovaPay AI Agent Training — Lab 4 Complete*
